<a href="https://colab.research.google.com/github/Meenakshi-N-24/RedFlag/blob/main/notebooks/03_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re, glob, json

MAX_CHARS = 700
HEADINGS = {"do's", "don'ts", "do's & don'ts for users"}

def norm(s):
    return s.replace("\u2019", "'").strip().lower()

def load_source(path):
    lines = open(path, encoding="utf-8").read().strip().splitlines()
    meta, body_start = {}, 0
    for i, line in enumerate(lines):
        m = re.match(r"\s*(Source|URL|Retrieved):\s*(.*)", line)
        if m:
            meta[m.group(1)] = m.group(2).strip()
        if len(meta) == 3:
            body_start = i + 1
            break
    assert len(meta) == 3, f"{path}: header is missing Source, URL or Retrieved"
    return meta, "\n".join(lines[body_start:]).strip()

def make_chunks(body):
    paras = [p.strip() for p in re.split(r"\n\s*\n", body) if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        force_new = norm(p) in HEADINGS
        if cur and (force_new or len(cur) + len(p) + 2 > MAX_CHARS):
            chunks.append(cur)
            cur = ""
        cur = (cur + "\n\n" + p) if cur else p
    if cur:
        chunks.append(cur)
    merged = []
    for c in chunks:
        if merged and len(merged[-1]) < 60:
            merged[-1] += "\n\n" + c
        else:
            merged.append(c)
    return merged

all_chunks = []
for path in sorted(glob.glob("*.txt")):
    meta, body = load_source(path)
    for n, text in enumerate(make_chunks(body), start=1):
        all_chunks.append({
            "id": f"{path[:-4]}-{n}",
            "source": meta["Source"],
            "url": meta["URL"],
            "retrieved": meta["Retrieved"],
            "text": text,
        })

json.dump(all_chunks, open("chunks.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

print("Total chunks:", len(all_chunks), "\n")
for c in all_chunks:
    print(c["id"], "|", len(c["text"]), "chars |", c["text"][:90].replace("\n", " "), "...")

Total chunks: 10 

chakshu-1 | 541 chars | Chakshu facilitates citizens to report the suspected or unsolicited communications receive ...
kerala_police_1930-1 | 460 chars | National Cyber Crime Reporting Helpline No 1930 August 03, 2023  National Cyber Crime Repo ...
kerala_police_1930-2 | 418 chars | The National Cybercrime Reporting Portal (https://cybercrime.gov.in) allows direct reporti ...
npci_fraud_awareness-1 | 391 chars | Fraudsters may use threatening tactics to entice gullible victims into making payments, su ...
npci_fraud_awareness-2 | 446 chars | To avoid falling for this type of scam, it's important to be cautious when receiving urgen ...
npci_fraud_awareness-3 | 282 chars | Remember, scammers often use fear and urgency as a tactic to pressurize victims into makin ...
npci_fraud_awareness-4 | 356 chars | Do's & Don'ts for users  Do's  Updated Mobile number is registered in bank records  UPI pi ...
npci_fraud_awareness-5 | 478 chars | Don'ts  Never forward any SMS on bein

In [3]:
!pip install -q sentence-transformers

import json, numpy as np
from sentence_transformers import SentenceTransformer

chunks = json.load(open("chunks.json", encoding="utf-8"))
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vecs = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)

def search(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_vecs @ q
    top = np.argsort(-scores)[:k]
    return [(chunks[i], float(scores[i])) for i in top]

tests = [
    "I got a message saying my electricity will be cut tonight unless I pay",
    "Where can I report a suspicious SMS?",
    "Someone asked for my UPI PIN so they could send me money",
    "I already lost money to an online fraud, what should I do?",
]
for q in tests:
    print("\nQ:", q)
    for c, s in search(q):
        print(f"  {s:.2f}  {c['id']}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Q: I got a message saying my electricity will be cut tonight unless I pay
  0.47  npci_fraud_awareness-1
  0.36  npci_fraud_awareness-2
  0.24  npci_fraud_awareness-3

Q: Where can I report a suspicious SMS?
  0.50  npci_fraud_awareness-2
  0.49  npci_fraud_awareness-1
  0.49  rbi_safe_digital_banking-1

Q: Someone asked for my UPI PIN so they could send me money
  0.55  npci_fraud_awareness-4
  0.48  npci_fraud_awareness-5
  0.46  npci_fraud_awareness-2

Q: I already lost money to an online fraud, what should I do?
  0.59  kerala_police_1930-2
  0.52  npci_fraud_awareness-3
  0.51  rbi_safe_digital_banking-2


In [4]:
def show_ranking(query):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_vecs @ q
    print("\nQ:", query)
    for rank, i in enumerate(np.argsort(-scores), start=1):
        print(f"  {rank:>2}. {scores[i]:.2f}  {chunks[i]['id']}")

show_ranking("Where can I report a suspicious SMS?")
show_ranking("How do I report a fraud call or message to the government?")
show_ranking("report suspected fraud communication SMS WhatsApp call")


Q: Where can I report a suspicious SMS?
   1. 0.50  npci_fraud_awareness-2
   2. 0.49  npci_fraud_awareness-1
   3. 0.49  rbi_safe_digital_banking-1
   4. 0.48  npci_fraud_awareness-5
   5. 0.44  chakshu-1
   6. 0.41  kerala_police_1930-2
   7. 0.38  kerala_police_1930-1
   8. 0.38  npci_fraud_awareness-4
   9. 0.32  npci_fraud_awareness-3
  10. 0.31  rbi_safe_digital_banking-2

Q: How do I report a fraud call or message to the government?
   1. 0.49  kerala_police_1930-2
   2. 0.46  npci_fraud_awareness-2
   3. 0.45  chakshu-1
   4. 0.43  npci_fraud_awareness-5
   5. 0.40  npci_fraud_awareness-1
   6. 0.37  npci_fraud_awareness-4
   7. 0.35  npci_fraud_awareness-3
   8. 0.35  rbi_safe_digital_banking-2
   9. 0.35  kerala_police_1930-1
  10. 0.25  rbi_safe_digital_banking-1

Q: report suspected fraud communication SMS WhatsApp call
   1. 0.57  chakshu-1
   2. 0.52  npci_fraud_awareness-1
   3. 0.52  npci_fraud_awareness-2
   4. 0.50  npci_fraud_awareness-5
   5. 0.43  rbi_safe_digital

In [5]:
eval_set = [
    ("A message says my electricity will be disconnected tonight unless I pay now", {"npci_fraud_awareness-1"}),
    ("How can I tell if an urgent payment message is genuine?", {"npci_fraud_awareness-2", "npci_fraud_awareness-3"}),
    ("Where do I report a scam SMS or WhatsApp message?", {"chakshu-1"}),
    ("Can I report a fake KYC update message?", {"chakshu-1"}),
    ("Is there a helpline for online financial fraud?", {"kerala_police_1930-2"}),
    ("What website can I use to file a cybercrime complaint?", {"kerala_police_1930-1", "kerala_police_1930-2"}),
    ("Do I enter my UPI PIN to receive money?", {"npci_fraud_awareness-4"}),
    ("Someone on the phone asked me to install a screen sharing app", {"npci_fraud_awareness-5"}),
    ("Should I share my debit card details with someone claiming to be from my bank?", {"npci_fraud_awareness-5"}),
    ("How do I find my bank's real contact number?", {"npci_fraud_awareness-4"}),
    ("I got an alert for a transaction I did not make", {"rbi_safe_digital_banking-2"}),
    ("What should I do if my debit card is stolen?", {"rbi_safe_digital_banking-2"}),
    ("Is it safe to do online banking on public wifi?", {"rbi_safe_digital_banking-1", "rbi_safe_digital_banking-2"}),
    ("Why should I report a financial fraud quickly?", {"kerala_police_1930-2"}),
    ("Who should I complain to if I was scammed through UPI?", {"npci_fraud_awareness-4", "kerala_police_1930-2"}),
    ("Can I track my cybercrime complaint?", {"kerala_police_1930-1"}),
]
off_topic = [
    "How do I get a refund for a defective phone?",
    "What is the best way to learn Python?",
    "How do I file my income tax return?",
]

top1 = top3 = 0
for q, ok in eval_set:
    ranked = search(q, k=3)
    ids = [c["id"] for c, s in ranked]
    hit1, hit3 = ids[0] in ok, any(i in ok for i in ids)
    top1 += hit1; top3 += hit3
    tag = "OK  " if hit1 else ("top3" if hit3 else "MISS")
    print(tag, f"{ranked[0][1]:.2f}", ids[0], "|", q[:55])

print(f"\nTop-1: {top1}/{len(eval_set)}   Top-3: {top3}/{len(eval_set)}")
print("\nOff-topic questions (best score):")
for q in off_topic:
    c, s = search(q, k=1)[0]
    print(f"  {s:.2f}  {c['id']}  | {q}")

OK   0.47 npci_fraud_awareness-1 | A message says my electricity will be disconnected toni
OK   0.70 npci_fraud_awareness-2 | How can I tell if an urgent payment message is genuine?
MISS 0.62 npci_fraud_awareness-2 | Where do I report a scam SMS or WhatsApp message?
top3 0.42 npci_fraud_awareness-2 | Can I report a fake KYC update message?
OK   0.75 kerala_police_1930-2 | Is there a helpline for online financial fraud?
OK   0.68 kerala_police_1930-2 | What website can I use to file a cybercrime complaint?
OK   0.54 npci_fraud_awareness-4 | Do I enter my UPI PIN to receive money?
OK   0.45 npci_fraud_awareness-5 | Someone on the phone asked me to install a screen shari
OK   0.59 npci_fraud_awareness-5 | Should I share my debit card details with someone claim
OK   0.44 npci_fraud_awareness-4 | How do I find my bank's real contact number?
OK   0.52 rbi_safe_digital_banking-2 | I got an alert for a transaction I did not make
top3 0.46 npci_fraud_awareness-5 | What should I do if my debit c

In [6]:
near_off_topic = [
    "What is the current RBI repo rate?",
    "How do I open a savings account?",
    "How do I check my credit score?",
    "How do I invest in mutual funds?",
]

def evaluate(model_name, query_prefix=""):
    emb = SentenceTransformer(model_name)
    vecs = emb.encode([c["text"] for c in chunks], normalize_embeddings=True)
    def srch(q, k=3):
        qv = emb.encode([query_prefix + q], normalize_embeddings=True)[0]
        s = vecs @ qv
        top = np.argsort(-s)[:k]
        return [(chunks[i]["id"], float(s[i])) for i in top]
    top1 = top3 = 0
    on_scores = []
    for q, ok in eval_set:
        r = srch(q)
        ids = [i for i, s in r]
        top1 += ids[0] in ok
        top3 += any(i in ok for i in ids)
        on_scores.append(r[0][1])
    off = [srch(q, 1)[0][1] for q in off_topic + near_off_topic]
    print(model_name)
    print(f"  top-1 {top1}/16   top-3 {top3}/16   lowest on-topic score {min(on_scores):.2f}   highest off-topic score {max(off):.2f}\n")

evaluate("all-MiniLM-L6-v2")
evaluate("BAAI/bge-small-en-v1.5", "Represent this sentence for searching relevant passages: ")
evaluate("all-mpnet-base-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

all-MiniLM-L6-v2
  top-1 10/16   top-3 15/16   lowest on-topic score 0.42   highest off-topic score 0.33



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

all-mpnet-base-v2
  top-1 9/16   top-3 11/16   lowest on-topic score 0.24   highest off-topic score 0.34

